In [12]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [13]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [14]:
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(24)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

wtch_dt_start:2026-07-20 15:33:04, wtch_dt_end:2026-07-21 15:33:04


In [15]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [16]:
@file:DependsOn("org.json:json:20250107")

In [17]:
import org.json.XML

fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}


In [18]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2144,1076,0,0.260000,59,4.977032,2.095003,0.250000,3.783583,5.450000,6.370000,14.218000
rtmWqChpla,Comparable<*>,2144,1329,0,,178,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,2144,1,0,,2144,null,null,,,,,
rtmWqWtchStaCd,String,2144,14,0,SEA1005,178,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,2144,2144,0,1,1,1072.500000,619.063809,1,536.416667,1072.500000,1608.583333,2144
rtmWqTu,Int,2144,150,0,5,189,24.481343,31.738470,0,5.000000,11.000000,32.000000,232
ph,Double,2144,126,0,7.500000,53,7.668997,0.262543,7.020000,7.490000,7.640000,7.910000,8.650000
rtmWqSlnty,Number,2144,1996,0,32.737999,4,21.826705,10.315607,0.284000,13.992000,26.645000,29.569000,34.032001
rtmWqCndctv,Float,2144,2069,0,45.320000,3,34.083227,15.459045,0.586000,23.511416,39.813000,45.167665,54.451000
rtmWqWtchDtlDt,String,2144,190,0,2026-07-20 15:40:00.0,14,null,null,2026-07-20 15:40:00.0,2026-07-20 21:30:00.0,2026-07-21 03:25:00.0,2026-07-21 09:25:00.0,2026-07-21 15:15:00.0


In [19]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert { rtmWqChpla }.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) 0.0 else value.toDouble()
}.convert {  colsOf<Number>() }.with { it.toString().trim().toDouble()}


df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2144,1076,0,0.260000,59,4.977032,2.095003,0.250000,3.783583,5.450000,6.370000,14.218000
rtmWqChpla,Double,2144,1329,0,0.000000,178,5.134701,5.196590,0.000000,1.310000,2.960000,7.900000,32.040000
rtmWqWtchStaCd,String,2144,14,0,SEA1005,178,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Double,2144,2144,0,1.000000,1,1072.500000,619.063809,1.000000,536.416667,1072.500000,1608.583333,2144.000000
rtmWqTu,Double,2144,150,0,5.000000,189,24.481343,31.738470,0.000000,5.000000,11.000000,32.000000,232.000000
ph,Double,2144,126,0,7.500000,53,7.668997,0.262543,7.020000,7.490000,7.640000,7.910000,8.650000
rtmWqSlnty,Double,2144,1996,0,32.737999,4,21.826705,10.315607,0.284000,14.003250,26.647000,29.573084,34.032001
rtmWqCndctv,Double,2144,2069,0,45.320000,3,34.083227,15.459045,0.586000,23.511417,39.813000,45.167667,54.451000
rtmWqWtchDtlDt,LocalDateTime,2144,190,0,2026-07-20T15:40,14,null,null,2026-07-20T15:40,2026-07-20T21:30,2026-07-21T03:25,2026-07-21T09:25,2026-07-21T15:15
rtmWtchWtem,Double,2144,760,0,26.990000,14,26.391460,2.333299,19.740000,24.994167,26.459999,28.180000,30.889999


In [20]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")

In [21]:
val renamedDf = df.rename(
    "rtmWqDoxn" to "용존산소",
    "rtmWqChpla" to "클로로필",
    "rtmWqWtchStaCd" to "관측정점코드",
    "num" to "순번",
    "rtmWqTu" to "탁도",
    "ph" to "수소이온농도",
    "rtmWqSlnty" to "염분",
    "rtmWqCndctv" to "전기전도도",
    "rtmWqWtchDtlDt" to "일시",
    "rtmWtchWtem" to "수온"
)
renamedDf

용존산소,클로로필,관측정점코드,순번,탁도,수소이온농도,염분,전기전도도,일시,수온
1.030000,0.000000,SEA1005,1.000000,7.000000,7.320000,3.540000,6.504000,2026-07-20T15:40,26.700001
6.730000,1.290000,SEA5001,2.000000,3.000000,7.690000,11.416000,20.367000,2026-07-20T15:40,29.730000
7.800000,1.810000,NEP1002,3.000000,18.000000,8.080000,1.244000,2.422000,2026-07-20T15:40,29.290001
6.040000,10.550000,SEA6001,4.000000,45.000000,7.850000,30.922001,47.544000,2026-07-20T15:40,25.850000
6.510000,13.380000,SEA7002,5.000000,24.000000,7.550000,26.721001,40.386000,2026-07-20T15:40,23.410000
7.388000,9.829000,SEA2005,6.000000,2.000000,7.930000,28.794001,44.579000,2026-07-20T15:40,30.510000
6.320000,2.360000,NEP2002,7.000000,6.000000,7.970000,27.841000,42.577000,2026-07-20T15:40,24.200001
5.007000,1.438000,NEP3001,8.000000,18.000000,7.690000,16.083000,26.316000,2026-07-20T15:40,27.700001
3.460000,2.831000,SEA2007,9.000000,50.000000,7.890000,31.962000,51.663000,2026-07-20T15:40,24.340000
6.764000,3.090000,NEP2001,10.000000,7.000000,7.870000,11.444000,19.269000,2026-07-20T15:40,27.730000


In [23]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1.000000,2026-07-20T15:40,1.030000,0.000000,SEA1005,7.000000,7.320000,3.540000,6.504000,26.700001
2.000000,2026-07-20T15:40,6.730000,1.290000,SEA5001,3.000000,7.690000,11.416000,20.367000,29.730000
3.000000,2026-07-20T15:40,7.800000,1.810000,NEP1002,18.000000,8.080000,1.244000,2.422000,29.290001
4.000000,2026-07-20T15:40,6.040000,10.550000,SEA6001,45.000000,7.850000,30.922001,47.544000,25.850000
5.000000,2026-07-20T15:40,6.510000,13.380000,SEA7002,24.000000,7.550000,26.721001,40.386000,23.410000
6.000000,2026-07-20T15:40,7.388000,9.829000,SEA2005,2.000000,7.930000,28.794001,44.579000,30.510000
7.000000,2026-07-20T15:40,6.320000,2.360000,NEP2002,6.000000,7.970000,27.841000,42.577000,24.200001
8.000000,2026-07-20T15:40,5.007000,1.438000,NEP3001,18.000000,7.690000,16.083000,26.316000,27.700001
9.000000,2026-07-20T15:40,3.460000,2.831000,SEA2007,50.000000,7.890000,31.962000,51.663000,24.340000
10.000000,2026-07-20T15:40,6.764000,3.090000,NEP2001,7.000000,7.870000,11.444000,19.269000,27.730000
